In [5]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.utils.data import DataLoader, Dataset
import torch

# 定义小型数据集
texts = [
    "I love this product!", "This is the worst experience ever", "I'm so happy with the service",
    "Horrible, I will never come back!", "Best purchase I've made in a long time", "I hate this!",
    "The quality is fantastic", "Totally disappointed with the results", "So much fun!", "Not what I expected"
]

labels = [1, 0, 1, 0, 1, 0, 1, 0, 1, 0]  # 1 -> Positive, 0 -> Negative

# 分割数据集
train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=0.3, random_state=42)

# 加载DistilBERT模型和Tokenizer
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# 定义数据集类
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# 创建训练和验证数据集
train_dataset = SentimentDataset(train_texts, train_labels, tokenizer)
val_dataset = SentimentDataset(val_texts, val_labels, tokenizer)

# 定义训练参数
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4)

# 使用Adam优化器
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# 训练模型
model.train()
epochs = 3
for epoch in range(epochs):
    for batch in train_loader:
        optimizer.zero_grad()

        # 前向传播
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels = batch['labels']
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        
        # 计算损失
        loss = outputs.loss
        loss.backward()
        
        # 更新参数
        optimizer.step()

    print(f"Epoch {epoch+1}/{epochs} completed")

# 模型评估
model.eval()
predictions = []
true_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels = batch['labels']
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        
        # 预测
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        
        predictions.extend(preds)
        true_labels.extend(labels.cpu().numpy())

# 打印所有预测结果
for i in range(len(predictions)):
    print(f"Text: {val_texts[i]}")
    print(f"True Label: {true_labels[i]}, Predicted Label: {predictions[i]}")
    print("------")

# 计算F1-Score
f1 = f1_score(true_labels, predictions)
print(f"F1-Score: {f1:.4f}")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/3 completed
Epoch 2/3 completed
Epoch 3/3 completed
Text: So much fun!
True Label: 1, Predicted Label: 1
------
Text: This is the worst experience ever
True Label: 0, Predicted Label: 1
------
Text: I hate this!
True Label: 0, Predicted Label: 1
------
F1-Score: 0.5000
